# 05 Predictive Modeling

This notebook trains and compares multiple models for predicting `Outcome`:

1. Logistic Regression
2. Regularized Logistic Regression
3. Random Forest
4. Gradient Boosting

The workflow is leakage-safe: patient identifiers are never used as predictors, repeated IDs are kept together in train/test splits, and all imputation, scaling, and one-hot encoding are performed inside scikit-learn `Pipeline` objects during cross-validation and final model fitting.

Outputs:

- `reports/model_comparison.csv`
- `models/best_model.joblib`

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=UserWarning)

# Make `src` importable when running from notebooks/ without `pip install -e .`.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "CardiacPatientData.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODELS_DIR, RAW_DATA_PATH, REPORTS_DIR
from src.evaluation import compute_metrics
from src.modeling import (
    RANDOM_STATE,
    build_feature_matrix,
    build_pipeline,
    build_preprocessor,
    make_leakage_safe_holdout,
    save_model,
    train_model,
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_COMPARISON_PATH = REPORTS_DIR / "model_comparison.csv"
BEST_MODEL_PATH = MODELS_DIR / "best_model.joblib"


## Load data

The raw dataset is loaded before splitting. Deterministic row-wise clinical features are added only after the train/test split so no train-set population statistics can leak into the held-out test set.

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(df.head())

outcome_distribution = (
    df["Outcome"].value_counts(normalize=True).sort_index().to_frame("proportion")
)
outcome_distribution["count"] = df["Outcome"].value_counts().sort_index()
display(outcome_distribution)


## Leakage safe train/test split

Repeated `ID` values indicate multiple rows per patient. If IDs repeat, a `StratifiedGroupKFold` split is used so all observations for the same patient stay entirely in either train or test. If IDs do not repeat, the notebook falls back to a standard stratified split.

In [ ]:
train_raw, test_raw, groups_train, split_strategy = make_leakage_safe_holdout(df)
overlapping_ids = set(train_raw["ID"]).intersection(test_raw["ID"])

print(f"Split strategy: {split_strategy}")
print(f"Train rows: {len(train_raw):,}; test rows: {len(test_raw):,}")
print(
    f"Train unique IDs: {train_raw['ID'].nunique():,}; "
    f"test unique IDs: {test_raw['ID'].nunique():,}"
)
print(f"Overlapping IDs across train/test: {len(overlapping_ids):,}")

display(
    pd.concat(
        {
            "train": train_raw["Outcome"].value_counts(normalize=True).sort_index(),
            "test": test_raw["Outcome"].value_counts(normalize=True).sort_index(),
        },
        axis=1,
    ).rename_axis("Outcome")
)


## Feature engineering and preprocessing

`ID` is removed from the modeling matrix to avoid memorization. Numerical and categorical preprocessing is defined inside a `ColumnTransformer`, which is then embedded in every model pipeline. This means imputation, scaling, and encoding are refit independently inside each cross-validation fold.

In [ ]:
X_train = build_feature_matrix(train_raw.drop(columns=["Outcome"]))
X_test = build_feature_matrix(test_raw.drop(columns=["Outcome"]))
y_train = train_raw["Outcome"].astype(int)
y_test = test_raw["Outcome"].astype(int)

preprocessor = build_preprocessor(X_train)
print(f"Feature matrix: {X_train.shape[1]} columns, {len(X_train):,} training rows")


## Cross-validation and model comparison

Each candidate model is wrapped in the shared preprocessing pipeline
(`src.modeling.build_pipeline`) and cross-validated with patient-grouped
`GroupKFold` via `src.modeling.train_model`. Imputation, scaling, and one-hot
encoding are therefore refit inside every fold, and no patient spans folds.
Models are ranked by **mean cross-validated AUROC**; held-out test metrics are
reported alongside for transparency but are not used for model selection.


In [ ]:
candidate_models = {
    "Logistic Regression": LogisticRegression(
        solver="lbfgs", max_iter=2000, random_state=RANDOM_STATE
    ),
    "Regularized Logistic Regression": LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        l1_ratio=0.5,
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
print(f"Comparing {len(candidate_models)} candidate models with patient-grouped CV.")


In [ ]:
results = []
fitted_models = {}

for model_name, model in candidate_models.items():
    print(f"Cross-validating {model_name}...")
    pipeline = build_pipeline(model, preprocessor)
    cv = train_model(pipeline, X_train, y_train, cv_groups=groups_train, n_splits=5)

    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline
    y_score = pipeline.predict_proba(X_test)[:, 1]
    test_metrics = compute_metrics(y_test, y_score, threshold=0.5)

    results.append(
        {
            "model": model_name,
            "cv_strategy": cv["cv"],
            "cv_mean_auroc": cv["metrics"]["auroc"]["mean"],
            "cv_std_auroc": cv["metrics"]["auroc"]["std"],
            "cv_mean_auprc": cv["metrics"]["auprc"]["mean"],
            "cv_mean_f1": cv["metrics"]["f1"]["mean"],
            "cv_mean_sensitivity": cv["metrics"]["sensitivity"]["mean"],
            "test_auroc": test_metrics["auroc"],
            "test_auprc": test_metrics["auprc"],
            "test_f1": test_metrics["f1"],
            "test_sensitivity": test_metrics["sensitivity"],
            "test_specificity": test_metrics["specificity"],
            "test_ppv": test_metrics["ppv"],
            "test_npv": test_metrics["npv"],
            "test_brier": test_metrics["brier_score"],
        }
    )

comparison_df = (
    pd.DataFrame(results)
    .sort_values(by=["cv_mean_auroc", "cv_mean_auprc"], ascending=False)
    .reset_index(drop=True)
)
comparison_df["rank_by_cv_auroc"] = np.arange(1, len(comparison_df) + 1)
comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)

print(f"Saved model comparison table to: {MODEL_COMPARISON_PATH}")
display(comparison_df)


## Save the best model

The best model is selected by held-out AUROC, with AUPRC and F1 used as tie-breakers. The saved object is the complete scikit-learn pipeline, so future scoring applies the same preprocessing steps before prediction.

In [ ]:
best_model_name = comparison_df.loc[0, "model"]
best_model = fitted_models[best_model_name]
save_model(best_model, BEST_MODEL_PATH)

print(f"Best model (by cross-validated AUROC): {best_model_name}")
print(f"Saved best model pipeline to: {BEST_MODEL_PATH}")
display(
    comparison_df.loc[
        0,
        [
            "model",
            "cv_mean_auroc",
            "test_auroc",
            "test_auprc",
            "test_f1",
            "test_sensitivity",
            "test_specificity",
            "test_ppv",
            "test_npv",
        ],
    ].to_frame("value")
)


## Interpretation, model performance and complexity

The cell below prints a short interpretation from the saved comparison table. In this workflow, added model complexity is considered justified only when a more complex model materially improves held-out AUROC/AUPRC over the simpler logistic baselines and does not sacrifice clinically important sensitivity or NPV.

In [ ]:
best = comparison_df.iloc[0]
logistic_mask = comparison_df["model"].isin(
    ["Logistic Regression", "Regularized Logistic Regression"]
)
logistic_best = comparison_df[logistic_mask].iloc[0]
auroc_gain = best["test_auroc"] - logistic_best["test_auroc"]
auprc_gain = best["test_auprc"] - logistic_best["test_auprc"]

if best["model"] == logistic_best["model"]:
    complexity_statement = (
        "The best-performing model is a logistic baseline, so additional "
        "tree-based complexity is not justified by this evaluation."
    )
elif auroc_gain >= 0.02 or auprc_gain >= 0.02:
    complexity_statement = (
        "The added complexity appears justified: the best non-logistic model "
        "improves AUROC or AUPRC by at least 0.02 over the best logistic baseline."
    )
else:
    complexity_statement = (
        "The added complexity is only weakly justified: the improvement over the "
        "best logistic baseline is small (<0.02 AUROC and AUPRC)."
    )

print(
    f"Best model: {best['model']} "
    f"(test AUROC={best['test_auroc']:.3f}, AUPRC={best['test_auprc']:.3f}, "
    f"F1={best['test_f1']:.3f}, sensitivity={best['test_sensitivity']:.3f}, "
    f"specificity={best['test_specificity']:.3f}, PPV={best['test_ppv']:.3f}, "
    f"NPV={best['test_npv']:.3f})."
)
print(
    f"Best logistic baseline: {logistic_best['model']} "
    f"(test AUROC={logistic_best['test_auroc']:.3f}, "
    f"AUPRC={logistic_best['test_auprc']:.3f})."
)
print(
    f"Difference vs best logistic baseline: "
    f"AUROC={auroc_gain:.3f}, AUPRC={auprc_gain:.3f}."
)
print(complexity_statement)


## Results summary

Models are ranked by mean cross-validated AUROC in
`reports/model_comparison.csv`, and the top model is refit on the full training
split and saved to `models/best_model.joblib`. The selected model balances
discrimination (AUROC/AUPRC) against clinically important sensitivity and NPV,
and the interpretation cell above flags whether any added tree-based complexity
is justified over the regularized logistic baseline. Re-running this notebook
regenerates both the comparison table and the saved model.
